# GUNB fieldmap modifications for `sc_inj` (Bmad)

In [1]:
from pytao import Tao
import pandas as pd
import numpy as np
import json
import os

In [2]:
THIN_OUTFILE = "$LCLS_LATTICE/bmad/master/gunb/gunb_thin_elements.bmad"
GUNB_OUTFILE = "$LCLS_LATTICE/bmad/master/gunb/gunb.bmad"
FIELDMAPS_OUTFILE = "$LCLS_LATTICE/bmad/master/gunb/gunb_fieldmaps.bmad"

## Reference element positions from `sc_diag0`

In [3]:
%%time

MODEL = '$LCLS_LATTICE/bmad/models/sc_diag0/'

tao = Tao(f'-init {MODEL}/tao.init -noplot -slice BEGINNING:ENDGUNB' )

def ele_info(ele):
    dat = tao.ele_head(ele)
    dat.update(tao.ele_gen_attribs(ele))
    return dat

def ele_table(match="*"):
    ix_ele = tao.lat_list(match, "ele.ix_ele", flags="-no_slaves")
    dat = list(map(ele_info, ix_ele))
    df = pd.DataFrame(dat, index=ix_ele)
    df.L.fillna(0, inplace=True)
    df['s_center'] = df['s'] - df['L']/2
    df['s_beginning'] = df['s'] - df['L']
    return  df

dat = ele_table()


CPU times: user 96.6 ms, sys: 8.6 ms, total: 105 ms
Wall time: 105 ms


## Thick elements

In [4]:
thick = dat[(dat.L != 0) & (dat.key != "Drift")]
thick.set_index('name', inplace=True)
thick

,universe,1^ix_branch,ix_ele,key,type,alias,descrip,is_on,s,s_start,...,Z_APERTURE_WIDTH2,units#Z_APERTURE_WIDTH2,Z_APERTURE_CENTER,units#Z_APERTURE_CENTER,PZ_APERTURE_WIDTH2,units#PZ_APERTURE_WIDTH2,PZ_APERTURE_CENTER,units#PZ_APERTURE_CENTER,s_center,s_beginning
name,,,,,,,,,,,,,,,,,,,,,
SOL1B,1,0,54,Solenoid,LBNL,SOLN:GUNB:212,,True,0.289580,0.203480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.246530,0.203480
BUN1B,1,0,25,Lcavity,LBNL,ACCL:GUNB:455,,True,0.918496,0.699736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.809116,0.699736
SOL2B,1,0,55,Solenoid,LBNL,SOLN:GUNB:823,,True,1.689260,1.602360,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.645810,1.602360


## Thin elements

In [5]:
# Thin elements
thin = dat[(dat.L == 0) & (dat.key != "Drift") & (dat.key != "Beginning_Ele") & (dat.s >= 0)]
thin = thin[~thin["name"].str.startswith("BEG")]
thin = thin[~thin["name"].str.startswith("END")]
thin = thin[~thin["name"].str.startswith("CATHODE")]
thin

,universe,1^ix_branch,ix_ele,key,name,type,alias,descrip,is_on,s,...,Z_APERTURE_WIDTH2,units#Z_APERTURE_WIDTH2,Z_APERTURE_CENTER,units#Z_APERTURE_CENTER,PZ_APERTURE_WIDTH2,units#PZ_APERTURE_WIDTH2,PZ_APERTURE_CENTER,units#PZ_APERTURE_CENTER,s_center,s_beginning
9,1,0,9,Multipole,SQ01B,solenoid trim,QUAD:GUNB:212:2,,True,0.246530,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.246530,0.246530
10,1,0,10,Multipole,CQ01B,solenoid trim,QUAD:GUNB:212:1,,True,0.246530,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.246530,0.246530
13,1,0,13,Instrument,VV01B,,,,True,0.387480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.387480,0.387480
15,1,0,15,Hkicker,XC01B,LBNL-xy,XCOR:GUNB:293,,True,0.484500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.484500,0.484500
16,1,0,16,Vkicker,YC01B,LBNL-xy,YCOR:GUNB:293,,True,0.484500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.484500,0.484500
18,1,0,18,Monitor,BPM1B,Stripline-4,BPMS:GUNB:314,,True,0.489650,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.489650,0.489650
20,1,0,20,Marker,IM01B,ICT,TORO:GUNB:360,,True,0.594692,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.594692,0.594692
22,1,0,22,Hkicker,XC02B,LBNL-xy,XCOR:GUNB:388,,True,0.670100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.670100,0.670100
23,1,0,23,Vkicker,YC02B,LBNL-xy,YCOR:GUNB:388,,True,0.670100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.670100,0.670100
27,1,0,27,Hkicker,XC03B,LBNL-xy,XCOR:GUNB:513,,True,0.964200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.964200,0.964200


In [6]:
def superimpose_lines(row):
    return f"""{row["name"]}[superimpose] = T
{row["name"]}[offset] = {row["s"]}
{row["name"]}[ref] = BEGGUNB
    """

lines = []
for index, row in thin.iterrows():
    #print(row.name, row["name"])
    lines.append(superimpose_lines(row))

with open(os.path.expandvars(THIN_OUTFILE), "w") as f:
    f.write('\n'.join(lines))

# New `GUNB` Line

In [7]:
L_gun = 0.175
L_buncher = 0.3
L_sol1 = thick.loc["SOL1B"].L

s_sol1 = thick.loc["SOL1B"].s_center
s_sol2 = thick.loc["SOL2B"].s_center
s_end = float(dat[dat["name"] == "ENDGUNB"].s)

L_pip1 =  np.round(float(dat[dat["name"] == "BUN1B"].s_center) - L_gun - L_buncher/2, 9)
L_pip2 =  np.round(s_end - L_buncher - L_pip1 - L_gun, 9)

In [8]:

def inherit_lines(child, parent, attribs):
    lines = [f"{child}[{a}] = {parent}[{a}]" for a in attribs]
    return '\n'.join(lines)
print(inherit_lines('A', 'B', [1,2,3]))

A[1] = B[1]
A[2] = B[2]
A[3] = B[3]


In [9]:
LINES = f"""

!
! LCLS GUNB for Bmad
! ------------------
!
! This is a drop-in replacement for the GUN line based on:
!     {MODEL}
!
! It is a skeleton on top of which fieldmaps and thin elements can be superimposed

GUNB_PIP01: PIPE, L = {L_pip1}
GUNB_PIP02: PIPE, L = {L_pip2}
BUN1B[L] = {L_buncher}

GUNB: Line = (BEGGUNB,
    RFGUNB, 
      GUNB_PIP01,
    BUN1B,
      GUNB_PIP02,
    ENDGUNB) ! Should end at s = {s_end} m


! From convert_to_astra notebook:
RFGUNB:  e_gun,
  !rf_frequency = 187000000.0,
  voltage = 759250.2223014082,
!  phi0 = 0.02361111111111111,
    L = 0.175


SOL1B[superimpose] = T
SOL1B[offset] = {s_sol1}
SOL1B[ref] = BEGGUNB

BUN1B[phi0] = -0.25
BUN1B[n_cell] = 2
BUN1B[longitudinal] = 0

SOL2B[superimpose] = T
SOL2B[offset] = {s_sol2}
SOL2B[ref] = BEGGUNB




"""


ATTRS = ["L", "field_calc", "grid_field", "tracking_method", "mat6_calc_method"]

LINES2 = f"""


! GUNB fieldmap modifications 
! ---------------------------

apex_gun: e_gun, 
   L = 0.175,
   tracking_method = time_runge_kutta,  mat6_calc_method = tracking, 
   rf_frequency = 1300e6/7, 
   field_calc = fieldmap, 
   grid_field = call::$LCLS_LATTICE/bmad/fieldmaps/apex_gun/apex_gun_fieldmesh.h5,
   aperture_at = continuous, aperture_type=wall3d

lcls2_solenoid: solenoid, 
   L =  0.48,
   bs_field = 0.06, ! T
   tracking_method = runge_kutta,  mat6_calc_method = tracking, 
   field_calc = fieldmap, 
   grid_field = call::$LCLS_LATTICE/bmad/fieldmaps/lcls2_solenoid/lcls2_solenoid_fieldmesh.h5    

apex_buncher_1300MHz: lcavity, 
    L = 0.3,
    voltage = 1e6,
    tracking_method = runge_kutta, 
    mat6_calc_method = tracking,  
    n_cell = 2, longitudinal = 0, 
    rf_frequency = 1300000000.0, 
    field_calc = fieldmap,
    grid_field = call::$LCLS_LATTICE/bmad/fieldmaps/apex_buncher/apex_buncher_1300MHz_center.h5



! RFGUNB
{inherit_lines("RFGUNB", "apex_gun", ATTRS+['rf_frequency'])}
RFGUNB[voltage] = 759250.2223014082
RFGUNB[phi0] = 0.02361111111111111


! SO1B
{inherit_lines("SOL1B", "lcls2_solenoid", ATTRS)}
! B_max = 0.0559 T,
! \int B dL = B_max * 0.12896517423475376 m,
! \int B^2 dL = B_max^2 * 0.08623994625099243 m,
! Hard edge L = 0.19285745050205194 m,
! Hard edge B = 0.03738073494674779 T
SOL1B[bs_field] = 0.0559

! BUN1B
{inherit_lines("BUN1B", "apex_buncher_1300MHz",  ATTRS)}
BUN1B[voltage] = 210652.95554408507
BUN1B[phi0] = -0.24583333333333332

! SOL2B
{inherit_lines("SOL2B", "lcls2_solenoid", ATTRS)}
! B_max = 0.0299 T,
! \int B dL = B_max * 0.12896517423475376 m,
! \int B^2 dL = B_max^2 * 0.08623994625099243 m,
! Hard edge L = 0.19285745050205194 m,
! Hard edge B = 0.019994346599423236 T,
SOL2B[bs_field] = 0.0299


"""


    
with open(os.path.expandvars(GUNB_OUTFILE), 'w') as f:
    f.write(LINES)
    
with open(os.path.expandvars(FIELDMAPS_OUTFILE), 'w') as f:
    f.write(LINES2)    
    
print(LINES)



!
! LCLS GUNB for Bmad
! ------------------
!
! This is a drop-in replacement for the GUN line based on:
!     $LCLS_LATTICE/bmad/models/sc_diag0/
!
! It is a skeleton on top of which fieldmaps and thin elements can be superimposed

GUNB_PIP01: PIPE, L = 0.484116
GUNB_PIP02: PIPE, L = 1.093461
BUN1B[L] = 0.3

GUNB: Line = (BEGGUNB,
    RFGUNB, 
      GUNB_PIP01,
    BUN1B,
      GUNB_PIP02,
    ENDGUNB) ! Should end at s = 2.052577 m


! From convert_to_astra notebook:
RFGUNB:  e_gun,
  !rf_frequency = 187000000.0,
  voltage = 759250.2223014082,
!  phi0 = 0.02361111111111111,
    L = 0.175


SOL1B[superimpose] = T
SOL1B[offset] = 0.24653
SOL1B[ref] = BEGGUNB

BUN1B[phi0] = -0.25
BUN1B[n_cell] = 2
BUN1B[longitudinal] = 0

SOL2B[superimpose] = T
SOL2B[offset] = 1.64581
SOL2B[ref] = BEGGUNB





